In [ ]:
# used in git bash

In [ ]:
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

curl -O $PREFIX/01-agentic-rag/code/ingest.py
curl -O $PREFIX/01-agentic-rag/code/rag_helper.py
curl -O $PREFIX/04-evaluation/code/evaluation_utils.py

In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc['course'] == 'llm-zoomcamp':
        documents_llm.append(doc)

len(documents_llm)

103

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url="https://models.github.ai/inference"
)


In [8]:
# testing with different model API

In [8]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url="https://models.github.ai/inference"
)

response = client.chat.completions.create(
    model="openai/gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ]
)

print(response.choices[0].message.content)

RateLimitError: Too many requests. For more on scraping GitHub and how it may affect your rights, please review our Terms of Service (https://docs.github.com/en/site-policy/github-terms/github-terms-of-service).

In [10]:
from pydantic import BaseModel
from openai import OpenAI
import os


class Person(BaseModel):
    name: str
    age: int


client = OpenAI(
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url="https://models.github.ai/inference"
)

response = client.beta.chat.completions.parse(
    model="openai/gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": "John is 25 years old."
        }
    ],
    response_format=Person
)

print(response.choices[0].message.parsed)

RateLimitError: Too many requests. For more on scraping GitHub and how it may affect your rights, please review our Terms of Service (https://docs.github.com/en/site-policy/github-terms/github-terms-of-service).

In [ ]:
# testing successful

In [ ]:
import json
user_prompt = json.dumps(doc)

In [ ]:
messages = [
    {'role': 'developer', 'content': data_gen_instructions},
    {'role': 'user', 'content': user_prompt}
]

In [ ]:
# BELOW THIS: this will not work we changed the mode and python code

In [14]:
response = openai_client.responses.parse(
    model="openai/gpt-4.1-mini",
    input=messages,
    text_format=Questions
)

NotFoundError: 404 page not found

In [13]:
response = openai_client.responses.create(
    model="openai/gpt-4.1-mini",
    input=messages
)

print(response.output_text)

NotFoundError: 404 page not found

In [12]:
result = response.output_parsed

print(result)

AttributeError: 'Response' object has no attribute 'output_parsed'

In [29]:
result = response.output_text
print(result)

You've just discovered the LLM Zoomcamp course and are wondering if you can still join. The answer is yes, you can still join the course. However, if you're interested in receiving a certificate, you'll need to submit your project before the submission deadline, while the course is still accepting submissions.


In [14]:
print(result.questions)

AttributeError: 'str' object has no attribute 'questions'

In [15]:
from evaluation_utils import llm_structured

In [16]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

You've just discovered the LLM Zoomcamp course and are wondering if you can still join. The answer is yes, you can still join the course. However, if you're interested in receiving a certificate, you'll need to submit your project before the submission deadline, while the course is still accepting submissions.


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [17]:
usage.input_tokens, usage.output_tokens

NameError: name 'usage' is not defined

In [17]:
# ABOVE THIS: not gonna work,,,,,,,,,,,,,,,,,

In [15]:
import importlib
import evaluation_utils

importlib.reload(evaluation_utils)

from evaluation_utils import llm_structured

In [16]:
import inspect
from evaluation_utils import llm_structured

print(inspect.getsource(llm_structured))

def llm_structured(
    client,
    instructions,
    user_prompt,
    output_type,
    model="openai/gpt-5",
):
    response = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "developer", 
                "content": instructions
            },
            {
                "role": "user", 
                "content": user_prompt
            },
        ],
        response_format=output_type,
    )

    return (
        response.choices[0].message.parsed,
        response.usage,
    )



In [17]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Just found llm-zoomcamp—can I enroll now or am I too late?', 'Starting late: can I still get the certificate? What’s required?', 'The cohort already started; can I hop in and be eligible for the cert if I submit a project?', 'Is late sign-up allowed? Anything special needed to qualify for the certificate?', 'Joined today—am I good to start, and how do certificates work for late starters?']


In [18]:
print(usage.model_dump())

{'completion_tokens': 1260, 'prompt_tokens': 210, 'total_tokens': 1470, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1152, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 13, 'engine_ttft_ms': 57, 'engine_ttlt_ms': 17888, 'pre_inference_ms': 145, 'service_tbt_ms': 13, 'service_ttft_ms': 468, 'service_ttlt_ms': 18292, 'total_duration_ms': 18155, 'user_visible_ttft_ms': 323}}


In [19]:
usage.prompt_tokens, usage.completion_tokens

(210, 1260)

In [20]:
from evaluation_utils import calc_price

In [21]:
cost = calc_price(usage)

cost

{'input_cost': 0.0001575,
 'output_cost': 0.0056700000000000006,
 'total_cost': 0.0058275}

In [22]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Just found llm-zoomcamp—can I enroll now or am I too late?',
  'document': '74eb249bbf'},
 {'question': 'Starting late: can I still get the certificate? What’s required?',
  'document': '74eb249bbf'},
 {'question': 'The cohort already started; can I hop in and be eligible for the cert if I submit a project?',
  'document': '74eb249bbf'},
 {'question': 'Is late sign-up allowed? Anything special needed to qualify for the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Joined today—am I good to start, and how do certificates work for late starters?',
  'document': '74eb249bbf'}]

In [23]:
import importlib
import evaluation_utils

importlib.reload(evaluation_utils)
from evaluation_utils import llm_structured_retry

In [24]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [25]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [26]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)